In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import transformers
import datasets
import torch
import pandas as pd
from tqdm import tqdm
import pickle
from transformer_lens import HookedTransformer, utils
import einops
import pickle
import os
from datetime import datetime
import lm_eval
from lm_eval import evaluate
from lm_eval.models.huggingface import HFLM

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

left_tokenizer = AutoTokenizer.from_pretrained("microsoft/Llama2-7b-WhoIsHarryPotter")
left_tokenizer.pad_token = left_tokenizer.eos_token
left_tokenizer.padding_side = "left"

right_tokenizer = AutoTokenizer.from_pretrained("microsoft/Llama2-7b-WhoIsHarryPotter")
right_tokenizer.pad_token = right_tokenizer.eos_token

lat_models = {}
# lat_models["WHP"] = AutoModelForCausalLM.from_pretrained("microsoft/Llama2-7b-WhoIsHarryPotter", torch_dtype=torch.bfloat16)
# lat_models["LLaMA"] = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-chat-hf", torch_dtype=torch.bfloat16)
# base_models = {"WHP": "microsoft/Llama2-7b-WhoIsHarryPotter", "LLaMA": "meta-llama/Llama-2-7b-chat-hf"}
base_models = {}

lat_model_names = {
    # "PCA_L8_Eps0.1": "models/hp-lat-llama-genericized_diff_hp_indices-epsilon=0.1-pgd_layer=82024-04-24-04-44-24",
    # "PCA_L8_Eps1": "models/hp-lat-llama-genericized_diff_hp_indices-2024-04-10-01-39-29",
    # "PCA_L8_Eps10": "models/hp-lat-llama-genericized_diff_hp_indices-2024-04-10-01-36-59",
    # "PCA_L15_Eps1": "models/hp-lat-llama-genericized_diff_hp_indices-epsilon=1.0-pgd_layer=152024-04-24-06-56-25",
    "PCA_L15_Eps10": "models/hp-lat-llama-genericized_diff_hp_indices-epsilon=10.0-pgd_layer=152024-04-24-06-57-07",
    "No_PCA_L8_Eps1": "models/hp-lat-llama-None-2024-04-10-16-09-25",
    "No_PCA_L8_Eps10": "models/hp-lat-llama-None-2024-04-10-16-09-25",
    "No_PCA_L15_Eps1": "models/hp-lat-llama-None-epsilon=1.0-pgd_layer=152024-04-24-06-54-54",
    "No_PCA_L15_Eps10": "models/hp-lat-llama-None-epsilon=10.0-pgd_layer=152024-04-24-06-55-04",
    "WHP_Replication": "models/hp-lat-llama-None-epsilon=0.0-pgd_layer=02024-04-24-03-28-01",
    "WHP_All_Coefs": "models/hp-lat-llama-None-epsilon=0.0-pgd_layer=82024-04-18-18-42-03",
}

merge_and_unload = False
# lat_model_names = {"WHP_L8_Eps1": "models/hp-lat-llama-None-2024-04-10-16-09-25", "SAQ_L8_Eps1": "models/hp-lat-llama-None-2024-04-03-09-29-58"}
# for short_name, model_name in lat_model_names.items():
#     lat_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-chat-hf", torch_dtype=torch.bfloat16)
#     lat_model = PeftModel.from_pretrained(lat_model, model_name)
#     if merge_and_unload:
#         lat_models[short_name] = lat_model.merge_and_unload()
#     else:
#         lat_models[short_name] = lat_model

save_dir = f"results/llama-lat-less-noise-2"
os.makedirs(save_dir, exist_ok=True)

In [3]:
capability_dict = {}
for model_name, model_path in base_models.items():
    print(f"Running on {model_name}")

    model = HFLM(pretrained=model_path, dtype=torch.bfloat16, device="cuda")
    results = lm_eval.simple_evaluate(
        model=model,
        tasks=["mmlu", "sciq"]
    )

    capability_dict[model_name] = results['results']
    with open(f"{save_dir}/full_capability_dict.pkl", "wb") as f:
        pickle.dump(capability_dict, f)

    del model
    print(f"Memory used: {torch.cuda.memory_allocated() / 1024**3}")

for model_name, model_path in lat_model_names.items():
    print(f"Running on {model_name}")

    model = HFLM(pretrained="meta-llama/Llama-2-7b-chat-hf", peft=model_path, dtype=torch.bfloat16, device="cuda")
    results = lm_eval.simple_evaluate(
        model=model,
        tasks=["mmlu", "sciq"]
    )

    capability_dict[model_name] = results['results']
    with open(f"{save_dir}/full_capability_dict.pkl", "wb") as f:
        pickle.dump(capability_dict, f)

    del model
    print(f"Memory used: {torch.cuda.memory_allocated() / 1024**3}")



Running on WHP


2024-05-02:13:30:28,698 WARNING  [logging.py:61] Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
2024-05-02:13:30:28,699 INFO     [huggingface.py:164] Using device 'cuda'
2024-05-02:13:31:02,810 INFO     [evaluator.py:131] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
2024-05-02:13:31:02,811 INFO     [evaluator.py:191] Using pre-initialized model
/data/phillip_guo/miniconda3/envs/hp-unlrn/lib/python3.10/site-packages/datasets/load.py:1429: FutureWarning: The repository for hails/mmlu_no_train contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/hails/mmlu_no_train
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this data

Memory used: 0.0079345703125
Running on LLaMA


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2024-05-02:13:41:17,813 INFO     [evaluator.py:131] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
2024-05-02:13:41:17,814 INFO     [evaluator.py:191] Using pre-initialized model
/data/phillip_guo/miniconda3/envs/hp-unlrn/lib/python3.10/site-packages/datasets/load.py:1429: FutureWarning: The repository for hails/mmlu_no_train contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/hails/mmlu_no_train
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(
2024-05-02:13:42:44,558 INFO     [task.py:395] Building contexts for sciq on rank 0...
100%|██████████| 1000/1000 [00:00<00:00, 1042.30it/s]
2024-05-02:13:42:45,559 INFO     [task.py:395] Building contexts for mmlu_world_religions on rank 0...
100%|█

Memory used: 0.0079345703125


In [4]:
torch.cuda.memory_allocated() // 1024**3

0

In [5]:
torch.cuda.empty_cache()
torch.cuda.memory_allocated() // 1024**3

0